In [ ]:
# For some reason,the loading of the models took around an hour this time. Not sure why this might be the case, it is definitely not the internet speed as the speed test suggests the connection strength is fine. Is it something to do with the HCC? Why is this the case. This was the same when the winter strom hit and we kept loosing power. 

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
import os
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr

2025-03-11 09:32:48.595384: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-11 09:33:17.993932: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-03-11 09:33:17.993997: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-03-11 09:33:25.441039: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-11 09:33:36.261854: I tensorflow/core/platform/cpu_feature_guar

In [2]:
# load the models required
frozen_model = tf.keras.models.load_model('models/VGG16_CNN_LSTM_frozen.keras')
finetuned_model = tf.keras.models.load_model('models/VGG16_CNN_LSTM_finetuned.keras')

2025-03-11 09:41:38.154283: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 31134 MB memory:  -> device: 0, name: Tesla V100S-PCIE-32GB, pci bus id: 0000:06:00.0, compute capability: 7.0


In [3]:
# load a test image to get the height and width information
file_path = 'All_data/block_0103/all_np_files/Block0103_2020_08_26.npy'

loaded_test_image = np.load(file_path)

print(loaded_test_image.shape)
image_height = loaded_test_image.shape[0]
image_width = loaded_test_image.shape[1]
print(image_height, image_width)

(768, 1024, 3)
768 1024


In [4]:
def get_all_preds_in_test_time_series(block_name, main_data_folder, model):
    # get the path to the block
    path_to_block = os.path.join(main_data_folder, block_name)
    # load the numpy file with the feature sub-widows
    test_features = np.load(os.path.join(path_to_block, [file for file in os.listdir(path_to_block) if file[:9] == 'subwindow'][0]))
    # get the predicted values
    all_predicted_values = model.predict(test_features)

    # return the predicted values
    return(all_predicted_values)

In [5]:
def prediction_on_test_data(pred_values, image_height, image_width, stride = 8, kernel_size = 32):
    # density map
    Density_map = np.zeros((image_height, image_width))

    # counts map
    counts_map = np.zeros((image_height, image_width))
    
    # now, for every window, we will keep adding the values together and also add the counts
    counter = 0
#     need a counter to move into each predicted value in the pred values list
    for ii in range(0, image_height, stride):
        for jj in range(0, image_width, stride):
#         operations for density map
#             get the window of interest
            new_window = Density_map[ii:ii + kernel_size,jj:jj+kernel_size]
#     fill each with the value c_k
            counts_window = np.full((new_window.shape[0], new_window.shape[1]), pred_values[counter])
#     get the shapes of this new window
            cw_height = counts_window.shape[0]
            cw_width = counts_window.shape[1]
#         Do c_k/r_2
            counts_window_new = counts_window/(cw_height*cw_width)
#     This is the value in the window now
            value_window = counts_window_new
#     place the values in the corrsponding area of the density map
            Density_map[ii:ii + kernel_size,jj:jj+kernel_size] = new_window + value_window

#         Let's now focus on capturing the counts of the windows
            new_window_c = counts_map[ii:ii + kernel_size,jj:jj+kernel_size]
#     get the counts area
            count = np.ones((new_window_c.shape[0], new_window_c.shape[1]))
#     keep adding the counts to reflect the addition of densities
            counts_map[ii:ii + kernel_size,jj:jj+kernel_size] = new_window_c + count
#     increase the counter
            counter = counter + 1
            
#         get the normalized count
    normalized_counts = np.divide(Density_map, counts_map)
    
#     entire count on the test set
    pred_on_test = np.sum(normalized_counts)
    
#     return the predicted value
    return(pred_on_test, normalized_counts)


In [6]:
def get_final_forecasted_and_true_values(preds_from_model, im_height, im_weight, stride, kernel_size, csv_file_name, block_name):
    final_preds_list = []
    for i in range(7):
        preds_per_image = prediction_on_test_data(preds_from_model[:,i], im_height, im_weight, stride , kernel_size)
        final_preds_list.append(preds_per_image[0])
    # make this list  a dataframe
    preds_df = pd.DataFrame(final_preds_list, columns = ['Forecasted_value'])
    
    # Where do we have the true values?
    true_val_location = 'All_data/test_true_counts'
    true_value_file = pd.read_csv(os.path.join(true_val_location, csv_file_name))
    
    # compute the mae
    mae_value = mean_absolute_error(true_value_file[['True_count']], preds_df[['Forecasted_value']])
    # compute the rmse
    rmse_value = np.sqrt(mean_squared_error(true_value_file[['True_count']], preds_df[['Forecasted_value']]))
    # pearsonr
    pearson_value = pearsonr(np.array(true_value_file[['True_count']]).reshape(-1), np.array(preds_df[['Forecasted_value']]).reshape(-1))
    # r2score
    r2score_value = r2_score(true_value_file[['True_count']], preds_df[['Forecasted_value']])
    # attach the true and the forecasted values together
    final_df = pd.concat((true_value_file, preds_df), axis = 1)
    # final df location
    final_loc = 'All_data/test_predicted_counts'
    # save this file
    final_df.to_csv(os.path.join(final_loc, block_name + '.csv'), index = False)
    all_metrics = [mae_value, rmse_value, pearson_value, r2score_value]

    return(final_preds_list, all_metrics, final_df)

Block 0103

Predictions wth the frozen model

In [7]:
# first get the predictions
frozen_preds_block_0103 = get_all_preds_in_test_time_series('block_0103', 'All_data', frozen_model)

2025-03-11 10:43:29.301624: W tensorflow/core/kernels/gpu_utils.cc:54] Failed to allocate memory for convolution redzone checking; skipping this check. This is benign and only means that we won't check cudnn for out-of-bounds reads and writes. This message will only be printed once.
2025-03-11 10:43:34.890382: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907


384/384 [==============================] - 269s 439ms/step


In [8]:
frozen_preds_block_0103.shape

(12288, 7)

In [9]:
frozen_final_forecasts_block_0103 = get_final_forecasted_and_true_values(frozen_preds_block_0103, image_height, image_width, 8, 32, 'true_counts_blk_0103.csv', 'frozen_block_0103_VGG16')

In [10]:
frozen_normalized_forecasts_block_0103 = frozen_final_forecasts_block_0103[0]

In [11]:
print(frozen_normalized_forecasts_block_0103)

[0.0, 49.15578523676109, 78.26549350564581, 0.0, 50.936753524072806, 50.25548877025399, 0.0]


In [12]:
mae_frozen_block_0103 = frozen_final_forecasts_block_0103[1]
mae_frozen_block_0103

[24.944788719533385,
 27.374734607827275,
 PearsonRResult(statistic=0.5580764810961975, pvalue=0.19294181332626675),
 -26.98736939603646]

In [13]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0103[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0103_2020_08_26,40,40.000661,0.000000
1,Block0103_2020_08_27,39,39.000001,49.155785
2,Block0103_2020_08_28,41,41.000000,78.265494
3,Block0103_2020_08_31,31,31.000000,0.000000
4,Block0103_2020_09_02,32,32.000000,50.936754
5,Block0103_2020_09_07,40,40.002086,50.255489
6,Block0103_2020_09_16,27,27.000176,0.000000


Predictions with the finetuned model

In [14]:
%%time
# first get the predictions
finetuned_preds_block_0103 = get_all_preds_in_test_time_series('block_0103', 'All_data', finetuned_model)

384/384 [==============================] - 169s 439ms/step
CPU times: user 3min 5s, sys: 9.6 s, total: 3min 15s
Wall time: 9min 58s


In [15]:
finetuned_preds_block_0103.shape

(12288, 7)

In [16]:
finetuned_final_forecasts_block_0103 = get_final_forecasted_and_true_values(finetuned_preds_block_0103, image_height, image_width, 8, 32, 'true_counts_blk_0103.csv', 'finetuned_block_0103_VGG16')

In [17]:
finetuned_normalized_forecasts_block_0103 = finetuned_final_forecasts_block_0103[0]

In [18]:
print(finetuned_normalized_forecasts_block_0103)

[37.0978417124324, 66.39048013720468, 76.69998319929691, 0.0, 43.20824792054801, 52.215098499863565, 1.9947531554756401]


In [19]:
mae_finetuned_block_0103 = finetuned_final_forecasts_block_0103[1]
mae_finetuned_block_0103

[20.774459269857875,
 23.58645358150738,
 PearsonRResult(statistic=0.8347761569983222, pvalue=0.01945678033176668),
 -19.777224721857863]

In [20]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0103[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0103_2020_08_26,40,40.000661,37.097842
1,Block0103_2020_08_27,39,39.000001,66.390480
2,Block0103_2020_08_28,41,41.000000,76.699983
3,Block0103_2020_08_31,31,31.000000,0.000000
4,Block0103_2020_09_02,32,32.000000,43.208248
5,Block0103_2020_09_07,40,40.002086,52.215098
6,Block0103_2020_09_16,27,27.000176,1.994753


Block 0104

Predictions wth the frozen model

In [21]:
%%time
# first get the predictions
frozen_preds_block_0104 = get_all_preds_in_test_time_series('block_0104', 'All_data', frozen_model)

384/384 [==============================] - 168s 439ms/step
CPU times: user 3min 5s, sys: 9.62 s, total: 3min 15s
Wall time: 9min 15s


In [22]:
frozen_preds_block_0104.shape

(12288, 7)

In [23]:
frozen_final_forecasts_block_0104 = get_final_forecasted_and_true_values(frozen_preds_block_0104, image_height, image_width, 8, 32, 'true_counts_blk_0104.csv', 'frozen_block_0104_VGG16')

In [24]:
frozen_normalized_forecasts_block_0104 = frozen_final_forecasts_block_0104[0]

In [25]:
print(frozen_normalized_forecasts_block_0104)

[0.0, 33.510852583044006, 59.93030766330551, 0.0, 42.91059840102713, 42.23511754962803, 0.0]


In [26]:
mae_frozen_block_0104 = frozen_final_forecasts_block_0104[1]
mae_frozen_block_0104

[18.512410885286382,
 24.033041336735447,
 PearsonRResult(statistic=0.46746599607743766, pvalue=0.29019366448717465),
 -23.39807475755905]

In [27]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0104[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0104_2020_08_26,33,33.000000,0.000000
1,Block0104_2020_08_27,30,30.000000,33.510853
2,Block0104_2020_08_28,39,39.000001,59.930308
3,Block0104_2020_08_31,40,40.000000,0.000000
4,Block0104_2020_09_02,41,40.998810,42.910598
5,Block0104_2020_09_07,42,42.169009,42.235118
6,Block0104_2020_09_16,30,30.005317,0.000000


Predictions with the finetuned model

In [28]:
# first get the predictions
finetuned_preds_block_0104 = get_all_preds_in_test_time_series('block_0104', 'All_data', finetuned_model)

384/384 [==============================] - 169s 440ms/step


In [29]:
finetuned_preds_block_0104.shape

(12288, 7)

In [30]:
finetuned_final_forecasts_block_0104 = get_final_forecasted_and_true_values(finetuned_preds_block_0104, image_height, image_width, 8, 32, 'true_counts_blk_0104.csv', 'finetuned_block_0104_VGG16')

In [31]:
finetuned_normalized_forecasts_block_0104 = finetuned_final_forecasts_block_0104[0]

In [32]:
print(finetuned_normalized_forecasts_block_0104)

[20.04942996589542, 34.99935809200709, 44.73825504577136, 0.0, 22.491674434838238, 30.50347363010079, 0.5050262344423876]


In [33]:
mae_finetuned_block_0104 = finetuned_final_forecasts_block_0104[1]
mae_finetuned_block_0104

[17.598286981785947,
 21.28151487539438,
 PearsonRResult(statistic=0.17070218847645208, pvalue=0.71441061913212),
 -18.13124215016377]

In [34]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0104[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0104_2020_08_26,33,33.000000,20.049430
1,Block0104_2020_08_27,30,30.000000,34.999358
2,Block0104_2020_08_28,39,39.000001,44.738255
3,Block0104_2020_08_31,40,40.000000,0.000000
4,Block0104_2020_09_02,41,40.998810,22.491674
5,Block0104_2020_09_07,42,42.169009,30.503474
6,Block0104_2020_09_16,30,30.005317,0.505026


Block 0105

Predictions wth the frozen model

In [36]:
# first get the predictions
frozen_preds_block_0105 = get_all_preds_in_test_time_series('block_0105', 'All_data', frozen_model)

384/384 [==============================] - 168s 440ms/step


In [37]:
frozen_preds_block_0105.shape

(12288, 7)

In [38]:
frozen_final_forecasts_block_0105 = get_final_forecasted_and_true_values(frozen_preds_block_0105, image_height, image_width, 8, 32, 'true_counts_blk_0105.csv', 'frozen_block_0105_VGG16')

In [39]:
frozen_normalized_forecasts_block_0105 = frozen_final_forecasts_block_0105[0]

In [40]:
print(frozen_normalized_forecasts_block_0105)

[0.0, 46.44513853841816, 74.26071695443164, 0.0, 50.15140486341259, 48.419334960034256, 0.0]


In [41]:
mae_frozen_block_0105 = frozen_final_forecasts_block_0105[1]
mae_frozen_block_0105

[20.325227902328095,
 24.821065811161713,
 PearsonRResult(statistic=0.6785690289631604, pvalue=0.09375186842109026),
 -5.500469442743114]

In [42]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0105[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0105_2020_08_26,40,40.000001,0.000000
1,Block0105_2020_08_27,46,46.002743,46.445139
2,Block0105_2020_08_28,58,58.000696,74.260717
3,Block0105_2020_08_31,41,41.000032,0.000000
4,Block0105_2020_09_02,41,41.001190,50.151405
5,Block0105_2020_09_07,36,36.000022,48.419335
6,Block0105_2020_09_16,23,23.000000,0.000000


Predictions with the finetuned model

In [43]:
# first get the predictions
finetuned_preds_block_0105 = get_all_preds_in_test_time_series('block_0105', 'All_data', finetuned_model)

384/384 [==============================] - 169s 440ms/step


In [44]:
finetuned_preds_block_0105.shape

(12288, 7)

In [45]:
finetuned_final_forecasts_block_0105 = get_final_forecasted_and_true_values(finetuned_preds_block_0105, image_height, image_width, 8, 32, 'true_counts_blk_0105.csv', 'finetuned_block_0105_VGG16')

In [46]:
finetuned_normalized_forecasts_block_0105 = finetuned_final_forecasts_block_0105[0]

In [47]:
print(finetuned_normalized_forecasts_block_0105)

[31.238760936129594, 55.95160404718839, 67.7631756534875, 0.0, 35.35721413374511, 44.71375442067377, 0.5807955702784966]


In [48]:
mae_finetuned_block_0105 = finetuned_final_forecasts_block_0105[1]
mae_finetuned_block_0105

[15.178823354456638,
 19.133010395592738,
 PearsonRResult(statistic=0.7556703254683818, pvalue=0.049436353809571114),
 -2.8625177116914573]

In [49]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0105[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0105_2020_08_26,40,40.000001,31.238761
1,Block0105_2020_08_27,46,46.002743,55.951604
2,Block0105_2020_08_28,58,58.000696,67.763176
3,Block0105_2020_08_31,41,41.000032,0.000000
4,Block0105_2020_09_02,41,41.001190,35.357214
5,Block0105_2020_09_07,36,36.000022,44.713754
6,Block0105_2020_09_16,23,23.000000,0.580796


Block 0106

Predictions wth the frozen model

In [50]:
# first get the predictions
frozen_preds_block_0106 = get_all_preds_in_test_time_series('block_0106', 'All_data', frozen_model)

384/384 [==============================] - 168s 440ms/step


In [51]:
frozen_preds_block_0106.shape

(12288, 7)

In [52]:
frozen_final_forecasts_block_0106 = get_final_forecasted_and_true_values(frozen_preds_block_0106, image_height, image_width, 8, 32, 'true_counts_blk_0106.csv', 'frozen_block_0106_VGG16')

In [53]:
frozen_normalized_forecasts_block_0106 = frozen_final_forecasts_block_0106[0]

In [54]:
print(frozen_normalized_forecasts_block_0106)

[0.0, 38.92099452039377, 65.60598337592653, 0.0, 45.64208464698297, 45.06840464079546, 0.0]


In [55]:
mae_frozen_block_0106 = frozen_final_forecasts_block_0106[1]
mae_frozen_block_0106

[20.4655241231029,
 26.739985189083143,
 PearsonRResult(statistic=0.7384992330160663, pvalue=0.05799863997002525),
 -58.993687650183055]

In [56]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0106[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0106_2020_08_26,39,38.999667,0.000000
1,Block0106_2020_08_27,39,38.999989,38.920995
2,Block0106_2020_08_28,45,45.000000,65.605983
3,Block0106_2020_08_31,40,39.986887,0.000000
4,Block0106_2020_09_02,43,42.999997,45.642085
5,Block0106_2020_09_07,48,47.996502,45.068405
6,Block0106_2020_09_16,38,38.000000,0.000000


Predictions with the finetuned model

In [57]:
# first get the predictions
finetuned_preds_block_0106 = get_all_preds_in_test_time_series('block_0106', 'All_data', finetuned_model)

384/384 [==============================] - 168s 440ms/step


In [58]:
finetuned_preds_block_0106.shape

(12288, 7)

In [59]:
finetuned_final_forecasts_block_0106 = get_final_forecasted_and_true_values(finetuned_preds_block_0106, image_height, image_width, 8, 32, 'true_counts_blk_0106.csv', 'finetuned_block_0106_VGG16')

In [60]:
finetuned_normalized_forecasts_block_0106 = finetuned_final_forecasts_block_0106[0]

In [61]:
print(finetuned_normalized_forecasts_block_0106)

[28.248601869293907, 50.02132628120414, 59.014881051962135, 0.0, 32.42386186290288, 40.80949868312176, 1.3492337135479224]


In [62]:
mae_finetuned_block_0106 = finetuned_final_forecasts_block_0106[1]
mae_finetuned_block_0106

[18.600715886328544,
 22.48909501491613,
 PearsonRResult(statistic=0.5410356176072092, pvalue=0.20981121631789912),
 -41.43529166936019]

In [63]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0106[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0106_2020_08_26,39,38.999667,28.248602
1,Block0106_2020_08_27,39,38.999989,50.021326
2,Block0106_2020_08_28,45,45.000000,59.014881
3,Block0106_2020_08_31,40,39.986887,0.000000
4,Block0106_2020_09_02,43,42.999997,32.423862
5,Block0106_2020_09_07,48,47.996502,40.809499
6,Block0106_2020_09_16,38,38.000000,1.349234


Block 0201

Predictions wth the frozen model

In [64]:
# first get the predictions
frozen_preds_block_0201 = get_all_preds_in_test_time_series('block_0201', 'All_data', frozen_model)

384/384 [==============================] - 168s 440ms/step


In [65]:
frozen_preds_block_0201.shape

(12288, 7)

In [66]:
frozen_final_forecasts_block_0201 = get_final_forecasted_and_true_values(frozen_preds_block_0201, image_height, image_width, 8, 32, 'true_counts_blk_0201.csv', 'frozen_block_0201_VGG16')

In [67]:
frozen_normalized_forecasts_block_0201 = frozen_final_forecasts_block_0201[0]

In [68]:
print(frozen_normalized_forecasts_block_0201)

[0.0, 41.238861999888556, 69.33516400294728, 0.0, 46.54934647792025, 46.198419888628344, 0.0]


In [69]:
mae_frozen_block_0201 = frozen_final_forecasts_block_0201[1]
mae_frozen_block_0201

[21.977724052801047,
 26.643430901638308,
 PearsonRResult(statistic=0.4711291772241795, pvalue=0.2859116530595703),
 -18.65183508491999]

In [70]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0201[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0201_2020_08_26,45,45.000217,0.000000
1,Block0201_2020_08_27,45,45.000040,41.238862
2,Block0201_2020_08_28,47,47.000001,69.335164
3,Block0201_2020_08_31,38,38.000000,0.000000
4,Block0201_2020_09_02,42,42.000041,46.549346
5,Block0201_2020_09_07,35,35.000000,46.198420
6,Block0201_2020_09_16,29,29.000000,0.000000


Predictions with the finetuned model

In [71]:
# first get the predictions
finetuned_preds_block_0201 = get_all_preds_in_test_time_series('block_0201', 'All_data', finetuned_model)

384/384 [==============================] - 169s 440ms/step


In [72]:
finetuned_preds_block_0201.shape

(12288, 7)

In [73]:
finetuned_final_forecasts_block_0201 = get_final_forecasted_and_true_values(finetuned_preds_block_0201, image_height, image_width, 8, 32, 'true_counts_blk_0201.csv', 'finetuned_block_0201_VGG16')

In [74]:
finetuned_normalized_forecasts_block_0201 = finetuned_final_forecasts_block_0201[0]

In [75]:
print(finetuned_normalized_forecasts_block_0201)

[28.681589861692835, 51.056368897483786, 63.40404012416198, 0.0, 32.776844819165035, 42.54244607095128, 1.0809508750314003]


In [76]:
mae_finetuned_block_0201 = finetuned_final_forecasts_block_0201[1]
mae_finetuned_block_0201

[17.351924219529682,
 20.485155023385357,
 PearsonRResult(statistic=0.6982599872512343, pvalue=0.0809964062755261),
 -10.617196180945982]

In [77]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0201[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0201_2020_08_26,45,45.000217,28.681590
1,Block0201_2020_08_27,45,45.000040,51.056369
2,Block0201_2020_08_28,47,47.000001,63.404040
3,Block0201_2020_08_31,38,38.000000,0.000000
4,Block0201_2020_09_02,42,42.000041,32.776845
5,Block0201_2020_09_07,35,35.000000,42.542446
6,Block0201_2020_09_16,29,29.000000,1.080951


Block 0202

Predictions wth the frozen model

In [78]:
# first get the predictions
frozen_preds_block_0202 = get_all_preds_in_test_time_series('block_0202', 'All_data', frozen_model)

384/384 [==============================] - 168s 439ms/step


In [79]:
frozen_preds_block_0202.shape

(12288, 7)

In [80]:
frozen_final_forecasts_block_0202 = get_final_forecasted_and_true_values(frozen_preds_block_0202, image_height, image_width, 8, 32, 'true_counts_blk_0202.csv', 'frozen_block_0202_VGG16')

In [81]:
frozen_normalized_forecasts_block_0202 = frozen_final_forecasts_block_0202[0]

In [82]:
print(frozen_normalized_forecasts_block_0202)

[0.0, 49.66856350549319, 77.81125178123514, 0.0, 50.64336219817374, 50.73295052747967, 0.0]


In [83]:
mae_frozen_block_0202 = frozen_final_forecasts_block_0202[1]
mae_frozen_block_0202

[28.97944685891168,
 31.31712180704487,
 PearsonRResult(statistic=0.6032295254816125, pvalue=0.15158291119846948),
 -285.05561783087484]

In [84]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0202[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0202_2020_08_26,18,17.999960,0.000000
1,Block0202_2020_08_27,21,21.000000,49.668564
2,Block0202_2020_08_28,23,23.000000,77.811252
3,Block0202_2020_08_31,21,20.999982,0.000000
4,Block0202_2020_09_02,21,21.000000,50.643362
5,Block0202_2020_09_07,18,18.000000,50.732951
6,Block0202_2020_09_16,18,18.000000,0.000000


Predictions with the finetuned model

In [85]:
# first get the predictions
finetuned_preds_block_0202 = get_all_preds_in_test_time_series('block_0202', 'All_data', finetuned_model)

384/384 [==============================] - 168s 440ms/step


In [86]:
finetuned_preds_block_0202.shape

(12288, 7)

In [87]:
finetuned_final_forecasts_block_0202 = get_final_forecasted_and_true_values(finetuned_preds_block_0202, image_height, image_width, 8, 32, 'true_counts_blk_0202.csv', 'finetuned_block_0202_VGG16')

In [88]:
finetuned_normalized_forecasts_block_0202 = finetuned_final_forecasts_block_0202[0]

In [89]:
print(finetuned_normalized_forecasts_block_0202)

[39.35898410901882, 70.6805991785518, 82.1516939216772, 0.0, 46.52980046144512, 54.59128009013723, 0.8905111401791146]


In [90]:
mae_finetuned_block_0202 = finetuned_final_forecasts_block_0202[1]
mae_finetuned_block_0202

[32.917406660093,
 36.14992220895292,
 PearsonRResult(statistic=0.45565818296943716, pvalue=0.30418762555715834),
 -380.1549220830598]

In [91]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0202[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0202_2020_08_26,18,17.999960,39.358984
1,Block0202_2020_08_27,21,21.000000,70.680599
2,Block0202_2020_08_28,23,23.000000,82.151694
3,Block0202_2020_08_31,21,20.999982,0.000000
4,Block0202_2020_09_02,21,21.000000,46.529800
5,Block0202_2020_09_07,18,18.000000,54.591280
6,Block0202_2020_09_16,18,18.000000,0.890511


Block 0205

Predictions wth the frozen model

In [92]:
# first get the predictions
frozen_preds_block_0205 = get_all_preds_in_test_time_series('block_0205', 'All_data', frozen_model)

384/384 [==============================] - 169s 440ms/step


In [93]:
frozen_preds_block_0205.shape

(12288, 7)

In [94]:
frozen_final_forecasts_block_0205 = get_final_forecasted_and_true_values(frozen_preds_block_0205, image_height, image_width, 8, 32, 'true_counts_blk_0205.csv', 'frozen_block_0205_VGG16')

In [95]:
frozen_normalized_forecasts_block_0205 = frozen_final_forecasts_block_0205[0]

In [96]:
print(frozen_normalized_forecasts_block_0205)

[0.0, 36.37075084084735, 62.7980981866535, 0.0, 44.22517568651773, 43.84342852869626, 0.0]


In [97]:
mae_frozen_block_0205 = frozen_final_forecasts_block_0205[1]
mae_frozen_block_0205

[21.499421651574302,
 27.236656738170133,
 PearsonRResult(statistic=0.3276855882941394, pvalue=0.47308321003284276),
 -44.211365725587704]

In [98]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0205[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0205_2020_08_26,44,44.000000,0.000000
1,Block0205_2020_08_27,42,42.000001,36.370751
2,Block0205_2020_08_28,45,45.000000,62.798098
3,Block0205_2020_08_31,43,43.000000,0.000000
4,Block0205_2020_09_02,39,39.000000,44.225176
5,Block0205_2020_09_07,41,40.999915,43.843429
6,Block0205_2020_09_16,32,31.999656,0.000000


Predictions with the finetuned model

In [99]:
# first get the predictions
finetuned_preds_block_0205 = get_all_preds_in_test_time_series('block_0205', 'All_data', finetuned_model)

384/384 [==============================] - 169s 440ms/step


In [100]:
finetuned_preds_block_0205.shape

(12288, 7)

In [101]:
finetuned_final_forecasts_block_0205 = get_final_forecasted_and_true_values(finetuned_preds_block_0205, image_height, image_width, 8, 32, 'true_counts_blk_0205.csv', 'finetuned_block_0205_VGG16')

In [102]:
finetuned_normalized_forecasts_block_0205 = finetuned_final_forecasts_block_0205[0]

In [103]:
print(finetuned_normalized_forecasts_block_0205)

[27.666472464259794, 50.045581363302816, 57.31592902771657, 0.0, 32.02209076398867, 38.99318627849153, 1.9997810926894695]


In [104]:
mae_finetuned_block_0205 = finetuned_final_forecasts_block_0205[1]
mae_finetuned_block_0205

[16.95428282736999,
 21.662770110938062,
 PearsonRResult(statistic=0.5350994865369857, pvalue=0.21584627589261715),
 -27.60013039190077]

In [105]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0205[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0205_2020_08_26,44,44.000000,27.666472
1,Block0205_2020_08_27,42,42.000001,50.045581
2,Block0205_2020_08_28,45,45.000000,57.315929
3,Block0205_2020_08_31,43,43.000000,0.000000
4,Block0205_2020_09_02,39,39.000000,32.022091
5,Block0205_2020_09_07,41,40.999915,38.993186
6,Block0205_2020_09_16,32,31.999656,1.999781


Block 0206

Predictions wth the frozen model

In [106]:
# first get the predictions
frozen_preds_block_0206 = get_all_preds_in_test_time_series('block_0206', 'All_data', frozen_model)

384/384 [==============================] - 169s 441ms/step


In [107]:
frozen_preds_block_0206.shape

(12288, 7)

In [108]:
frozen_final_forecasts_block_0206 = get_final_forecasted_and_true_values(frozen_preds_block_0206, image_height, image_width, 8, 32, 'true_counts_blk_0206.csv', 'frozen_block_0206_VGG16')

In [109]:
frozen_normalized_forecasts_block_0206 = frozen_final_forecasts_block_0206[0]

In [110]:
print(frozen_normalized_forecasts_block_0206)

[0.0, 45.124245254597305, 73.80786236230497, 0.0, 48.43430677135699, 48.46034118789894, 0.0]


In [111]:
mae_frozen_block_0206 = frozen_final_forecasts_block_0206[1]
mae_frozen_block_0206

[25.40382222516546,
 27.897116845146595,
 PearsonRResult(statistic=0.18071417141548385, pvalue=0.6981949708931502),
 -8.838546771237445]

In [112]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0206[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0206_2020_08_26,41,40.999838,0.000000
1,Block0206_2020_08_27,42,41.998810,45.124245
2,Block0206_2020_08_28,39,39.000067,73.807862
3,Block0206_2020_08_31,32,32.000003,0.000000
4,Block0206_2020_09_02,25,25.000352,48.434307
5,Block0206_2020_09_07,23,23.000040,48.460341
6,Block0206_2020_09_16,18,18.000000,0.000000


Predictions with the finetuned model

In [113]:
# first get the predictions
finetuned_preds_block_0206 = get_all_preds_in_test_time_series('block_0206', 'All_data', finetuned_model)

384/384 [==============================] - 170s 443ms/step


In [114]:
finetuned_preds_block_0206.shape

(12288, 7)

In [115]:
finetuned_final_forecasts_block_0206 = get_final_forecasted_and_true_values(finetuned_preds_block_0206, image_height, image_width, 8, 32, 'true_counts_blk_0206.csv', 'finetuned_block_0206_VGG16')

In [116]:
finetuned_normalized_forecasts_block_0206 = finetuned_final_forecasts_block_0206[0]

In [117]:
print(finetuned_normalized_forecasts_block_0206)

[32.87385865408113, 60.122841335150184, 72.2007404695535, 0.0, 37.37255432526793, 47.8932767804702, 1.7088350147693443]


In [118]:
mae_finetuned_block_0206 = finetuned_final_forecasts_block_0206[1]
mae_finetuned_block_0206

[20.71524560594162,
 22.548101297753703,
 PearsonRResult(statistic=0.5248132131893034, pvalue=0.2264957434381312),
 -5.4273546786775935]

In [119]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0206[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0206_2020_08_26,41,40.999838,32.873859
1,Block0206_2020_08_27,42,41.998810,60.122841
2,Block0206_2020_08_28,39,39.000067,72.200740
3,Block0206_2020_08_31,32,32.000003,0.000000
4,Block0206_2020_09_02,25,25.000352,37.372554
5,Block0206_2020_09_07,23,23.000040,47.893277
6,Block0206_2020_09_16,18,18.000000,1.708835


Block 0302

Predictions wth the frozen model

In [120]:
# first get the predictions
frozen_preds_block_0302 = get_all_preds_in_test_time_series('block_0302', 'All_data', frozen_model)

384/384 [==============================] - 169s 442ms/step


In [121]:
frozen_preds_block_0302.shape

(12288, 7)

In [122]:
frozen_final_forecasts_block_0302 = get_final_forecasted_and_true_values(frozen_preds_block_0302, image_height, image_width, 8, 32, 'true_counts_blk_0302.csv', 'frozen_block_0302_VGG16')

In [123]:
frozen_normalized_forecasts_block_0302 = frozen_final_forecasts_block_0302[0]

In [124]:
print(frozen_normalized_forecasts_block_0302)

[0.0, 44.923177035207715, 73.32561881443895, 0.0, 48.94388078825109, 48.23598127441557, 0.0]


In [125]:
mae_frozen_block_0302 = frozen_final_forecasts_block_0302[1]
mae_frozen_block_0302

[23.654614834556842,
 30.927300547376007,
 PearsonRResult(statistic=0.42410271172087066, pvalue=0.3429833024994166),
 -35.61593596737382]

In [126]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0302[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0302_2020_08_26,49,49.000000,0.000000
1,Block0302_2020_08_27,49,49.000005,44.923177
2,Block0302_2020_08_28,54,53.999570,73.325619
3,Block0302_2020_08_31,50,50.042349,0.000000
4,Block0302_2020_09_02,43,43.000009,48.943881
5,Block0302_2020_09_07,48,48.000572,48.235981
6,Block0302_2020_09_16,37,36.999999,0.000000


Predictions with the finetuned model

In [127]:
# first get the predictions
finetuned_preds_block_0302 = get_all_preds_in_test_time_series('block_0302', 'All_data', finetuned_model)

384/384 [==============================] - 169s 441ms/step


In [128]:
finetuned_preds_block_0302.shape

(12288, 7)

In [129]:
finetuned_final_forecasts_block_0302 = get_final_forecasted_and_true_values(finetuned_preds_block_0302, image_height, image_width, 8, 32, 'true_counts_blk_0302.csv', 'finetuned_block_0302_VGG16')

In [130]:
finetuned_normalized_forecasts_block_0302 = finetuned_final_forecasts_block_0302[0]

In [131]:
print(finetuned_normalized_forecasts_block_0302)

[31.72452901710389, 56.50789650363544, 67.57361299061645, 0.0, 36.55044425541236, 45.747287059273674, 1.5071604570618242]


In [132]:
mae_finetuned_block_0302 = finetuned_final_forecasts_block_0302[1]
mae_finetuned_block_0302

[18.936012672200018,
 24.915475744220366,
 PearsonRResult(statistic=0.5846848764026179, pvalue=0.16797472669091235),
 -22.764270036313157]

In [133]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0302[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0302_2020_08_26,49,49.000000,31.724529
1,Block0302_2020_08_27,49,49.000005,56.507897
2,Block0302_2020_08_28,54,53.999570,67.573613
3,Block0302_2020_08_31,50,50.042349,0.000000
4,Block0302_2020_09_02,43,43.000009,36.550444
5,Block0302_2020_09_07,48,48.000572,45.747287
6,Block0302_2020_09_16,37,36.999999,1.507160


Block 0303

Predictions wth the frozen model

In [134]:
# first get the predictions
frozen_preds_block_0303 = get_all_preds_in_test_time_series('block_0303', 'All_data', frozen_model)

384/384 [==============================] - 169s 441ms/step


In [135]:
frozen_preds_block_0303.shape

(12288, 7)

In [136]:
frozen_final_forecasts_block_0303 = get_final_forecasted_and_true_values(frozen_preds_block_0303, image_height, image_width, 8, 32, 'true_counts_blk_0303.csv', 'frozen_block_0303_VGG16')

In [137]:
frozen_normalized_forecasts_block_0303 = frozen_final_forecasts_block_0303[0]

In [138]:
print(frozen_normalized_forecasts_block_0303)

[0.0, 44.11691221319345, 72.45919655691465, 0.0, 48.35050941749551, 47.35675219108817, 0.0]


In [139]:
mae_frozen_block_0303 = frozen_final_forecasts_block_0303[1]
mae_frozen_block_0303

[23.721363707472126,
 28.294121511097327,
 PearsonRResult(statistic=0.22332793120942868, pvalue=0.6302502740870275),
 -11.769306084684988]

In [140]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0303[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0303_2020_08_26,49,49.000171,0.000000
1,Block0303_2020_08_27,46,46.000007,44.116912
2,Block0303_2020_08_28,43,42.999982,72.459197
3,Block0303_2020_08_31,39,38.999993,0.000000
4,Block0303_2020_09_02,36,36.025884,48.350509
5,Block0303_2020_09_07,36,36.000000,47.356752
6,Block0303_2020_09_16,23,23.000000,0.000000


Predictions with the finetuned model

In [141]:
# first get the predictions
finetuned_preds_block_0303 = get_all_preds_in_test_time_series('block_0303', 'All_data', finetuned_model)

384/384 [==============================] - 169s 442ms/step


In [142]:
finetuned_preds_block_0303.shape

(12288, 7)

In [143]:
finetuned_final_forecasts_block_0303 = get_final_forecasted_and_true_values(finetuned_preds_block_0303, image_height, image_width, 8, 32, 'true_counts_blk_0303.csv', 'finetuned_block_0303_VGG16')

In [144]:
finetuned_normalized_forecasts_block_0303 = finetuned_final_forecasts_block_0303[0]

In [145]:
print(finetuned_normalized_forecasts_block_0303)

[31.59610666918328, 55.85301029511159, 66.2604620386136, 0.0, 36.31400553889908, 44.489524567698545, 0.9778181786044419]


In [146]:
mae_finetuned_block_0303 = finetuned_final_forecasts_block_0303[1]
mae_finetuned_block_0303

[17.191868227505015,
 20.76801637214035,
 PearsonRResult(statistic=0.57251617267232, pvalue=0.1791829434275935),
 -5.879627180221677]

In [147]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0303[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0303_2020_08_26,49,49.000171,31.596107
1,Block0303_2020_08_27,46,46.000007,55.853010
2,Block0303_2020_08_28,43,42.999982,66.260462
3,Block0303_2020_08_31,39,38.999993,0.000000
4,Block0303_2020_09_02,36,36.025884,36.314006
5,Block0303_2020_09_07,36,36.000000,44.489525
6,Block0303_2020_09_16,23,23.000000,0.977818


Block 0304

Predictions wth the frozen model

In [148]:
# first get the predictions
frozen_preds_block_0304 = get_all_preds_in_test_time_series('block_0304', 'All_data', frozen_model)

384/384 [==============================] - 169s 441ms/step


In [149]:
frozen_preds_block_0304.shape

(12288, 7)

In [150]:
frozen_final_forecasts_block_0304 = get_final_forecasted_and_true_values(frozen_preds_block_0304, image_height, image_width, 8, 32, 'true_counts_blk_0304.csv', 'frozen_block_0304_VGG16')

In [151]:
frozen_normalized_forecasts_block_0304 = frozen_final_forecasts_block_0304[0]

In [152]:
print(frozen_normalized_forecasts_block_0304)

[0.0, 40.52165668903403, 68.2355242015841, 0.0, 46.617998066130305, 45.96055518992944, 0.0]


In [153]:
mae_frozen_block_0304 = frozen_final_forecasts_block_0304[1]
mae_frozen_block_0304

[21.327488681229973,
 25.532570977845577,
 PearsonRResult(statistic=0.4335494624307317, pvalue=0.3311597410320111),
 -16.825723692074437]

In [154]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0304[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0304_2020_08_26,37,37.000000,0.000000
1,Block0304_2020_08_27,41,41.002057,40.521657
2,Block0304_2020_08_28,43,42.999998,68.235524
3,Block0304_2020_08_31,42,41.999955,0.000000
4,Block0304_2020_09_02,38,38.000000,46.617998
5,Block0304_2020_09_07,34,34.000000,45.960555
6,Block0304_2020_09_16,24,24.000000,0.000000


Predictions with the finetuned model

In [155]:
# first get the predictions
finetuned_preds_block_0304 = get_all_preds_in_test_time_series('block_0304', 'All_data', finetuned_model)

384/384 [==============================] - 169s 440ms/step


In [156]:
finetuned_preds_block_0304.shape

(12288, 7)

In [157]:
finetuned_final_forecasts_block_0304 = get_final_forecasted_and_true_values(finetuned_preds_block_0304, image_height, image_width, 8, 32, 'true_counts_blk_0304.csv', 'finetuned_block_0304_VGG16')

In [158]:
finetuned_normalized_forecasts_block_0304 = finetuned_final_forecasts_block_0304[0]

In [159]:
print(finetuned_normalized_forecasts_block_0304)

[28.505354669051535, 50.213224001750795, 60.139471885534334, 0.0, 32.5439800575942, 41.058773687135265, 0.9624848599750444]


In [160]:
mae_finetuned_block_0304 = finetuned_final_forecasts_block_0304[1]
mae_finetuned_block_0304

[16.057092855399944,
 20.089533649302012,
 PearsonRResult(statistic=0.5074613372236065, pvalue=0.24500428244802397),
 -10.035646623926036]

In [161]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0304[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0304_2020_08_26,37,37.000000,28.505355
1,Block0304_2020_08_27,41,41.002057,50.213224
2,Block0304_2020_08_28,43,42.999998,60.139472
3,Block0304_2020_08_31,42,41.999955,0.000000
4,Block0304_2020_09_02,38,38.000000,32.543980
5,Block0304_2020_09_07,34,34.000000,41.058774
6,Block0304_2020_09_16,24,24.000000,0.962485


Block 0305

Predictions wth the frozen model

In [162]:
# first get the predictions
frozen_preds_block_0305 = get_all_preds_in_test_time_series('block_0305', 'All_data', frozen_model)

384/384 [==============================] - 169s 441ms/step


In [163]:
frozen_preds_block_0305.shape

(12288, 7)

In [164]:
frozen_final_forecasts_block_0305 = get_final_forecasted_and_true_values(frozen_preds_block_0305, image_height, image_width, 8, 32, 'true_counts_blk_0305.csv', 'frozen_block_0305_VGG16')

In [165]:
frozen_normalized_forecasts_block_0305 = frozen_final_forecasts_block_0305[0]

In [166]:
print(frozen_normalized_forecasts_block_0305)

[0.0, 51.517943806827034, 80.4729729087572, 0.0, 51.82669325655702, 51.40206684883258, 0.0]


In [167]:
mae_frozen_block_0305 = frozen_final_forecasts_block_0305[1]
mae_frozen_block_0305

[27.888525260139122,
 30.15136113056651,
 PearsonRResult(statistic=0.1817275039210909, pvalue=0.6965587286700499),
 -15.449824343894388]

In [168]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0305[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0305_2020_08_26,46,46.000000,0.000000
1,Block0305_2020_08_27,35,35.000000,51.517944
2,Block0305_2020_08_28,37,37.000000,80.472973
3,Block0305_2020_08_31,30,30.000652,0.000000
4,Block0305_2020_09_02,35,35.001190,51.826693
5,Block0305_2020_09_07,29,28.999994,51.402067
6,Block0305_2020_09_16,20,20.000018,0.000000


Predictions with the finetuned model

In [169]:
# first get the predictions
finetuned_preds_block_0305 = get_all_preds_in_test_time_series('block_0305', 'All_data', finetuned_model)

384/384 [==============================] - 169s 442ms/step


In [170]:
finetuned_preds_block_0305.shape

(12288, 7)

In [171]:
finetuned_final_forecasts_block_0305 = get_final_forecasted_and_true_values(finetuned_preds_block_0305, image_height, image_width, 8, 32, 'true_counts_blk_0305.csv', 'finetuned_block_0305_VGG16')

In [172]:
finetuned_normalized_forecasts_block_0305 = finetuned_final_forecasts_block_0305[0]

In [173]:
print(finetuned_normalized_forecasts_block_0305)

[42.0789234732512, 73.95559218900819, 82.05849572414557, 0.0, 48.940053648073864, 57.70440192124757, 2.5086126577358905]


In [174]:
mae_finetuned_block_0305 = finetuned_final_forecasts_block_0305[1]
mae_finetuned_block_0305

[25.438715335926876,
 28.753626994501797,
 PearsonRResult(statistic=0.524613339606391, pvalue=0.22670506793142808),
 -13.960037740623404]

In [175]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0305[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0305_2020_08_26,46,46.000000,42.078923
1,Block0305_2020_08_27,35,35.000000,73.955592
2,Block0305_2020_08_28,37,37.000000,82.058496
3,Block0305_2020_08_31,30,30.000652,0.000000
4,Block0305_2020_09_02,35,35.001190,48.940054
5,Block0305_2020_09_07,29,28.999994,57.704402
6,Block0305_2020_09_16,20,20.000018,2.508613


Block 0306

Predictions wth the frozen model

In [176]:
# first get the predictions
frozen_preds_block_0306 = get_all_preds_in_test_time_series('block_0306', 'All_data', frozen_model)

384/384 [==============================] - 169s 441ms/step


In [177]:
frozen_preds_block_0306.shape

(12288, 7)

In [178]:
frozen_final_forecasts_block_0306 = get_final_forecasted_and_true_values(frozen_preds_block_0306, image_height, image_width, 8, 32, 'true_counts_blk_0306.csv', 'frozen_block_0306_VGG16')

In [179]:
frozen_normalized_forecasts_block_0306 = frozen_final_forecasts_block_0306[0]

In [180]:
print(frozen_normalized_forecasts_block_0306)

[0.0, 41.5936680725564, 69.39280086991484, 0.0, 46.65571308155388, 46.83547581666629, 0.0]


In [181]:
mae_frozen_block_0306 = frozen_final_forecasts_block_0306[1]
mae_frozen_block_0306

[20.925379691527343,
 25.46047717888839,
 PearsonRResult(statistic=0.412394150407963, pvalue=0.35787966027994955),
 -8.827833852307588]

In [182]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0306[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0306_2020_08_26,41,41.000009,0.000000
1,Block0306_2020_08_27,41,41.000003,41.593668
2,Block0306_2020_08_28,43,43.006851,69.392801
3,Block0306_2020_08_31,40,39.997604,0.000000
4,Block0306_2020_09_02,40,40.000000,46.655713
5,Block0306_2020_09_07,33,32.999982,46.835476
6,Block0306_2020_09_16,18,18.000000,0.000000


Predictions with the finetuned model

In [183]:
# first get the predictions
finetuned_preds_block_0306 = get_all_preds_in_test_time_series('block_0306', 'All_data', finetuned_model)

384/384 [==============================] - 169s 441ms/step


In [184]:
finetuned_preds_block_0306.shape

(12288, 7)

In [185]:
finetuned_final_forecasts_block_0306 = get_final_forecasted_and_true_values(finetuned_preds_block_0306, image_height, image_width, 8, 32, 'true_counts_blk_0306.csv', 'finetuned_block_0306_VGG16')

In [186]:
finetuned_normalized_forecasts_block_0306 = finetuned_final_forecasts_block_0306[0]

In [187]:
print(finetuned_normalized_forecasts_block_0306)

[31.11435444488657, 56.18500935644549, 67.28386236006261, 0.0, 35.567732714796875, 45.70401936021941, 1.5249655090268184]


In [188]:
mae_finetuned_block_0306 = finetuned_final_forecasts_block_0306[1]
mae_finetuned_block_0306

[17.566548344002467,
 20.59986245096313,
 PearsonRResult(statistic=0.5544424815282054, pvalue=0.1964821926775441),
 -5.4335898257832405]

In [189]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0306[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0306_2020_08_26,41,41.000009,31.114354
1,Block0306_2020_08_27,41,41.000003,56.185009
2,Block0306_2020_08_28,43,43.006851,67.283862
3,Block0306_2020_08_31,40,39.997604,0.000000
4,Block0306_2020_09_02,40,40.000000,35.567733
5,Block0306_2020_09_07,33,32.999982,45.704019
6,Block0306_2020_09_16,18,18.000000,1.524966
